In [14]:
import random
import copy

FORWARD_VALUES = [25, 50, 75]
TURN_VALUES = [15, 30, 45]
LETTERS = ["A", "B", "C", "D"]


def parse_action(action_str):
    """解析字符串为结构化动作块"""
    blocks = [x.strip() for x in action_str.split(",")]
    parsed = []

    for b in blocks:
        if b.startswith("forward"):
            val = int(b.split()[1])
            parsed.append(("forward", val))
        elif b.startswith("turn left"):
            val = int(b.split()[2])
            parsed.append(("turn_left", val))
        elif b.startswith("turn right"):
            val = int(b.split()[2])
            parsed.append(("turn_right", val))
        elif b == "stop":
            parsed.append(("stop", 0))
    return parsed


def format_action(blocks):
    """结构化动作转回字符串"""
    strs = []
    for t, v in blocks:
        if t == "forward":
            strs.append(f"forward {v} cm")
        elif t == "turn_left":
            strs.append(f"turn left {v} degree")
        elif t == "turn_right":
            strs.append(f"turn right {v} degree")
        elif t == "stop":
            strs.append("stop")
    return ", ".join(strs)


def modify_value(blocks):
    """修改某个块的数值"""
    new_blocks = copy.deepcopy(blocks)
    idx = random.randrange(len(new_blocks))
    t, v = new_blocks[idx]

    if t == "forward":
        candidates = [x for x in FORWARD_VALUES if x != v]
        if candidates:
            new_blocks[idx] = ("forward", random.choice(candidates))

    elif t in ["turn_left", "turn_right"]:
        candidates = [x for x in TURN_VALUES if x != v]
        if candidates:
            new_blocks[idx] = (t, random.choice(candidates))

    return new_blocks


def swap_type(blocks):
    """替换动作类型"""
    new_blocks = copy.deepcopy(blocks)
    idx = random.randrange(len(new_blocks))
    t, v = new_blocks[idx]

    if t == "turn_left":
        new_blocks[idx] = ("turn_right", v)

    elif t == "turn_right":
        new_blocks[idx] = ("turn_left", v)

    elif t == "forward":
        new_type = random.choice(["turn_left", "turn_right"])
        new_blocks[idx] = (new_type, random.choice(TURN_VALUES))

    elif t in ["turn_left", "turn_right"]:
        new_blocks[idx] = ("forward", random.choice(FORWARD_VALUES))

    return new_blocks


def insert_block(blocks):
    """插入新动作块（不超过3块）"""
    if len(blocks) >= 3:
        return blocks

    new_blocks = copy.deepcopy(blocks)

    candidate_types = ["forward", "turn_left", "turn_right"]
    t = random.choice(candidate_types)

    if t == "forward":
        v = random.choice(FORWARD_VALUES)
    else:
        v = random.choice(TURN_VALUES)

    pos = random.randint(0, len(new_blocks))
    new_blocks.insert(pos, (t, v))

    return new_blocks


def delete_block(blocks):
    """删除一个块（至少保留1个）"""
    if len(blocks) <= 1:
        return blocks

    new_blocks = copy.deepcopy(blocks)
    idx = random.randrange(len(new_blocks))
    del new_blocks[idx]
    return new_blocks


def generate_negative(blocks, original_str):
    """生成一个合法且不同的错误action"""
    strategies = [modify_value, swap_type, insert_block, delete_block]

    for _ in range(20):  # 最多尝试20次避免死循环
        strategy = random.choice(strategies)
        new_blocks = strategy(blocks)
        new_str = format_action(new_blocks)
        if new_str != original_str:
            return new_str

    return original_str  # 极端情况fallback


def create_choice(action):
    """
    输入:
        action: 正确action字符串
    输出:
        choices: 4个选项列表
        correct_letter: 正确答案字母 A/B/C/D
    """

    parsed = parse_action(action)

    choices = [action]
    negatives = set()

    while len(negatives) < 3:
        neg = generate_negative(parsed, action)
        if neg != action:
            negatives.add(neg)

    choices.extend(list(negatives))
    random.shuffle(choices)

    correct_index = choices.index(action)
    correct_letter = LETTERS[correct_index]

    return choices, correct_letter

# action = "forward 25 cm, turn left 30 degree"

# choices, ans = create_choice(action)

# for l, c in zip(["A","B","C","D"], choices):
#     print(l, c)

# print("Correct:", ans)
# print(choices)

In [ ]:
import json
import os
import random
import re
import copy
from tqdm import tqdm


forward_distance = 25
turn_angle = 15

def noise_action(action_id):
    p = random.random()
    if p < 0.7 and action_id != 0:
        return random.randint(1,3)
    else:
        return action_id
def action_id_to_str(action_id):
    # id: 0-stop, 1 move forward, 2 turn left, 3 turn right
    if action_id == 0:
        return "stop"
    elif action_id == 1:
        return f"forward {forward_distance} cm"
    elif action_id == 2:
        return f"turn left {turn_angle} degree"
    elif action_id == 3:
        return f"turn right {turn_angle} degree"
    else:
        raise ValueError(f"Invalid action ID: {action_id}")

def combine(action1,action2):
    idx = action1.rfind(', ')
    subaction0 = action1[:idx+1]+' ' if action1[:idx+1] != '' else action1[:idx+1]
    subaction1 = action1[idx+1:]
    match1 = re.search(r'-?\d+', subaction1)
    match1 = int(match1.group())
    match2 = re.search(r'-?\d+', action2)
    match2 = int(match2.group())
    if "forward" in subaction1:
        if match1+match2 <= 3*forward_distance:
            return f"{subaction0}forward {match1+match2} cm"
        else:
            return None
    elif "turn left" in subaction1:
        if match1+match2 <= 3*turn_angle:
            return f"{subaction0}turn left {match1+match2} degree"
        else:
            return None
    elif "turn right" in subaction1:
        if match1+match2 <= 3*turn_angle:
            return f"{subaction0}turn right {match1+match2} degree"
        else:
            return None
    else:
        raise ValueError(f"Invalid action: {action1}")

def spilt_method_1(x):
    return x.split("_")[1].split(".")[0]

def spilt_method_2(x):
    return x.split(".")[0]


config = {
    'r2r': {
        'image_path': '/nvme-ssd1/zwy/data/images_downsample/R2R_train',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/r2r.jsonl',
        'split_method': spilt_method_1,
    },
    'rxr': {
        'image_path': '/nvme-ssd1/zwy/data/images_downsample/RxR_train',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/rxr.jsonl',
        'split_method': spilt_method_1,
    },
    'streamvln_rxr': {
        'image_path': '/nvme-ssd1/zwy/data/streamvln/RxR/',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/streamvln_rxr.jsonl',
        'split_method': spilt_method_2,
        'sub_image_path':' rgb',
    },
    'streamvln_rxr_single_instruction': {
        'image_path': '/nvme-ssd1/zwy/data/streamvln/RxR/',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/streamvln_rxr_single_instruction.jsonl',
        'split_method': spilt_method_2,
        'sub_image_path':' rgb',
    },
    'envdrop': {
        'image_path': '/nvme-ssd1/zwy/data/images_downsample/R2R_envdrop',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/envdrop.jsonl',
        'split_method': spilt_method_1,
    },
    'scale_vln': {
        'image_path': '/nvme-ssd1/zwy/data/images_downsample/scale_vln',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/scale_vln.jsonl',
        'split_method': spilt_method_2,
    },
    'dagger': {
        'image_path': '/nvme-ssd1/zwy/data/images_downsample/dagger',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/dagger.jsonl',
        'split_method': spilt_method_1,
    },
    'dagger_wo_rxr': {
        'image_path': '/nvme-ssd1/zwy/data/images_downsample/dagger',
        'annotation_path': '/nvme-ssd1/zwy/lam_vln/data/sub_dataset/dagger_wo_rxr.jsonl',
        'split_method': spilt_method_1,
    }
}


def main(selected_subset_list, system_prompt, prompt_template, output_path, task_type):
    data2save = []
    for subset in tqdm(selected_subset_list):

        subset_config = config[subset]
        image_path = subset_config['image_path']
        annotation_path = subset_config['annotation_path']
        split_method = subset_config['split_method']
        sub_image_path = config.get('sub_image_path', None)

        # read annotations
        annotation = []
        with open(annotation_path, 'r') as f:
            for line in f:
                item = json.loads(line)
                annotation.append(item)
            
        for episode_item in annotation:
            episode_id = episode_item['episode_id']
            video_id = episode_item['video_id']
            instruction = episode_item['instruction']
            actions = episode_item['actions']
            assert actions[-1] == 0

            episode_image_path = os.path.join(image_path, str(video_id))
            if sub_image_path is not None:
                episode_image_path = os.path.join(episode_image_path, sub_image_path)
            episode_image_list = os.listdir(episode_image_path)
            episode_image_list = sorted(episode_image_list, key=lambda x: int(split_method(x)))
            episode_image_list = [episode_image_path+'/'+image for image in episode_image_list]
            if len(episode_image_list) != len(actions) + 1:
                episode_image_list.append(episode_image_list[-1])
            assert len(episode_image_list) == len(actions) + 1

            tmp_data = {
                "system": system_prompt,
                "conversations": [],
                "action_history":[],
                "episode_id":str(episode_id),
                # 'idm_input': [],
                "task type": task_type,
            }
            if task_type == "vln":
                formated_instruction = prompt_template.format(instruction)
                tmp_data["conversations"].append({"from": "user", "value": formated_instruction,"image":[episode_image_list[0]]})
                tmp_data["conversations"].append({"from": "assistant", "value": action_id_to_str(actions[0])}) 
                last_action = actions[0]
                pending_rgb_list = []
                for i in range(1, len(actions)):
                    prob = random.random()
                    if prob <= 0.7 and actions[i] == last_action and combine(tmp_data["conversations"][-1]['value'],action_id_to_str(last_action)) is not None:
                        tmp_data["conversations"][-1]['value'] = combine(tmp_data["conversations"][-1]['value'],action_id_to_str(last_action))
                        pending_rgb_list.append(episode_image_list[i])
                    else:
                        count = tmp_data["conversations"][-1]['value'].count(',')
                        if count < 2:
                            tmp_data["conversations"][-1]['value'] += ', ' + action_id_to_str(actions[i])
                            pending_rgb_list.append(episode_image_list[i])
                        else:
                            # tmp_data['idm_input'] = [tmp_data["conversations"][0]['image'][-1], episode_image_list[i]]
                            data2save.append(copy.deepcopy(tmp_data))
                            tmp_data["action_history"].append(tmp_data["conversations"][1]['value'])
                            while len(pending_rgb_list)!= 0:
                                item = pending_rgb_list.pop(0)
                                tmp_data["conversations"][0]['image'].append(item)
                            tmp_data["conversations"][0]['image'].append(episode_image_list[i])
                            # while len(tmp_data["conversations"][0]['image']) > max_images:
                            #     tmp_data["conversations"][0]['image'] = tmp_data["conversations"][0]['image'][1:]
                            tmp_data["conversations"][1]['value'] = action_id_to_str(actions[i])
                            # assert len(conversation["action_history"]) == len(conversation["conversations"][0]['image']) - 1
                    last_action = actions[i]
                # tmp_data['idm_input'] = [tmp_data["conversations"][0]['image'][-1], episode_image_list[i+1]]
                data2save.append(copy.deepcopy(tmp_data))
            elif task_type == "idm_selection":
                formated_instruction = prompt_template
                tmp_data["conversations"].append({"from": "user", "value": formated_instruction,"image":[episode_image_list[0], episode_image_list[1]]})
                tmp_data["conversations"].append({"from": "assistant", "value": action_id_to_str(actions[0])}) 
                last_action = actions[0]
                for i in range(1, len(actions)):
                    prob = random.random()
                    if prob <= 0.7 and actions[i] == last_action and combine(tmp_data["conversations"][-1]['value'],action_id_to_str(last_action)) is not None:
                        tmp_data["conversations"][-1]['value'] = combine(tmp_data["conversations"][-1]['value'],action_id_to_str(last_action))
                        tmp_data["conversations"][-2]['image'][-1] = episode_image_list[i+1]
                    else:
                        count = tmp_data["conversations"][-1]['value'].count(',')
                        if count < 2:
                            tmp_data["conversations"][-1]['value'] += ', ' + action_id_to_str(actions[i])
                            tmp_data["conversations"][-2]['image'][-1] = episode_image_list[i+1]
                        else:
                            choices, ans = create_choice(tmp_data["conversations"][1]['value'])
                            tmp_data["conversations"][0]['value'] = prompt_template.format(*choices)
                            tmp_data["conversations"][1]['value'] = "The answer is " + ans
                            data2save.append(copy.deepcopy(tmp_data))
                            tmp_data["action_history"].append(tmp_data["conversations"][1]['value'])
                            tmp_data["conversations"][0]['image'] = [episode_image_list[i],episode_image_list[i+1]]
                            tmp_data["conversations"][1]['value'] = action_id_to_str(actions[i])
                            # assert len(conversation["action_history"]) == len(conversation["conversations"][0]['image']) - 1
                    last_action = actions[i]
                if tmp_data["conversations"][1]['value'] != 'stop':
                    data2save.append(copy.deepcopy(tmp_data))

    print(f"total number of samples = {len(data2save)}")

    with open(output_path, 'w', encoding='utf-8') as f:
        for item in data2save:
            json_line = json.dumps(item, ensure_ascii=False)
            f.write(json_line + '\n')
    print(len(data2save))

system_prompt = "You are a helpful assistant."
vln_prompt_template = "Imagine you are a robot programmed for navigation tasks. "\
    "You have been given a video of historical observations and an image of the current observation. "\
    "Your assigned task is: '{}'. Analyze this series of images to decide your next move, "\
    "which could involve turning left or right by a specific degree or moving forward a certain distance."

idm_prompt_template = "Imagine you are a robot programmed for navigation tasks. "\
    "You have been given an image of current view and an image of the goal view. "\
    "Analyze the two images to predict the navigation action from four choices that would move the robot from the current viewpoint to the goal view. "\
    "Select the correct answer from the four options (A, B, C, or D). "\
    "Options: \n A. {} \n B. {} \n C. {} \n D. {}"

selected_subset_list = ['r2r', 'dagger_wo_rxr', 'streamvln_rxr_single_instruction']
selected_subset_list = ['r2r', 'dagger_wo_rxr', 'streamvln_rxr']

output_path = f'/nvme-ssd1/zwy/navida/data/r2r_rxr_vln_data.jsonl'

task_type = "vln"

if task_type == "vln":
    prompt_template = vln_prompt_template
elif task_type == "idm_selection":
    prompt_template = idm_prompt_template

main(selected_subset_list, system_prompt, prompt_template, output_path, task_type)

100%|██████████| 3/3 [00:46<00:00, 15.59s/it]


total number of samples = 865547
865547


In [1]:
import json
import random
paths = [
         '/nvme-ssd1/zwy/navida/data/r2r_rxr_vln_data.jsonl',
         '/nvme-ssd1/zwy/navida/data/r2r_rxr_idm_selection_data.jsonl',
         ]
output_path = '/nvme-ssd1/zwy/navida/data/r2r_rxr_mix_data_with_idm_selection.jsonl'
data = []
for i, path in enumerate(paths):
    current_num = len(data)
    with open(path, 'r') as f:
        for line in f:
            item = json.loads(line)
            data.append(item)
    print(f"number of samples in {i+1} dataset = {len(data) - current_num}")

# random.shuffle(data1)
with open(output_path, 'w', encoding='utf-8') as f:
    for item in data:
        json_line = json.dumps(item, ensure_ascii=False)
        f.write(json_line + '\n')
print(f"whole samples in dataset = {len(data)}")

number of samples in 1 dataset = 865547
number of samples in 2 dataset = 578213
whole samples in dataset = 1443760
